# שיעור 09 - תבנית עיצוב מטא-קוגניציה


## הגדרה

מחברת זו מדגימה את תבנית העיצוב של מטה-קוגניציה באמצעות Microsoft Agent Framework.

**דרישות מוקדמות:**
- פריסת Azure OpenAI המוגדרת באמצעות משתני סביבה
- Azure CLI מאומת (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## מהי מטה-קוגניציה?

מטה-קוגניציה היא **חשיבה על חשיבה**. בהקשר של סוכני בינה מלאכותית, זה אומר לבנות סוכנים שיכולים:

- **להרהר** על התוצרים שלהם ועל תהליך ההסקה שלהם
- **לזהות שגיאות** ולהתאושש באופן אלגנטי במקום לכשל בשתיקה
- **להעריך** האם התשובות שלהם שלמות ומועילות
- **להתאים** את האסטרטגיה שלהם כאשר גישה ראשונית לא עובדת (למשל, לעבור למערכת גיבוי)

סוכן מטה-קוגניטיבי לא רק עונה על שאלות — הוא מנטר את הביצועים שלו ומתאים את עצמו תוך כדי פעולה.


## כלים ראשיים וכלי גיבוי

דפוס נפוץ במטא-קוגניציה הוא ה**אסטרטגיית הגיבוי**. הסוכן מנסה קודם כל כלי ראשי; אם הוא נכשל (למשל, שגיאת 404), הסוכן מזהה את הכשל ומעביר באופן שקוף לכלי גיבוי.

זה משקף מערכות בעולם האמיתי שבהן שירותים ראשיים עשויים להיות בלתי זמינים והסוכנים חייבים לאבחן בעצמם את הבעיה לפני שהם בוחרים מסלול חלופי.

להלן נגדיר שני כלי חיפוש טיסות:
- **ראשי** — כולל פריז, טוקיו וברצלונה
- **גיבוי** — כולל ברלין, סידני וניו יורק


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## סוכן המתבונן בעצמו עם התאוששות משגיאות

הסוכן שלהלן הונחה לנסות תחילה את מערכת הטיסה הראשית, לזהות כשלים, ולחזור באופן שקוף למערכת הגיבוי. לאחר כל תגובה הוא מעריך בקצרה האם השיב במלואו על שאלת המשתמש.


In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## תבנית הערכה עצמית

פן נוסף של מטה-קוגניציה הוא **הערכה עצמית**: סוכן נפרד (או אותו סוכן במעבר שני) בוחן תשובה מבחינת שלמות, דיוק ושימושיות.

להלן אנו יוצרים סוכן `ResponseEvaluator` שמדרג תגובות של סוכן הנסיעות בשלושה ממדים.


In [ ]:
evaluation_agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## סיכום

בשיעור הזה למדת כיצד לבנות **סוכנים מטא-קוגניטיביים** באמצעות Microsoft Agent Framework:

- **התבוננות עצמית**: סוכנים שמנטרים את ההסקה שלהם ומתקשרים בצורה שקופה מה קרה.
- **התאוששות משגיאות עם גיבויים**: תבנית של כלי ראשי + גיבוי שבה הסוכן מזהה כשלים (למשל, שגיאות 404) ומנסה אוטומטית מקור חלופי.
- **הערכה עצמית**: סוכן מעריך נפרד שמדרג תשובות לפי שלמות, דיוק ומועילות.

דפוסים אלה הופכים את הסוכנים לעמידים יותר, לשקופים ולאמינים — תכונות קריטיות לפריסות בסביבת ייצור.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
הבהרה:
מסמך זה תורגם באמצעות שירות תרגום מבוסס בינה מלאכותית Co-op Translator (https://github.com/Azure/co-op-translator). אף שאנו שואפים לדיוק, יש לשים לב כי תרגומים אוטומטיים עלולים להכיל שגיאות או אי־דיוקים. יש להתייחס למסמך המקורי בשפתו כאל המקור הסמכותי. במקרה של מידע קריטי מומלץ להיעזר בתרגום מקצועי אנושי. איננו אחראים לכל אי־הבנה או פרשנות שגויה הנובעת משימוש בתרגום זה.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
